# WACV Track 2 — train pipeline (Colab / Kaggle)

Generated by `scripts/export_colab_notebook.py`. Cells call the repo's real
scripts (`download-dataset`, `train`, `evaluate`, `package-submission`);
do not paste a second Trainer here.

**Secrets:** put Kaggle/Drive/Codabench tokens in the host secret store or
environment variables — never hardcode them in this notebook.

## Setup — locate repo root

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# Prefer an already-checked-out clone (Kaggle Dataset, Colab Drive mount, or
# local). Override with env REPO_ROOT if needed.
CANDIDATES = [
    Path(os.environ["REPO_ROOT"]) if os.environ.get("REPO_ROOT") else None,
    Path.cwd(),
    Path.cwd() / "attribute-based-person-retrieval-core",
    Path("/kaggle/working/attribute-based-person-retrieval-core"),
    Path("/content/attribute-based-person-retrieval-core"),
]


def find_repo() -> Path:
    for candidate in CANDIDATES:
        if candidate is None:
            continue
        marker = candidate / "scripts" / "train.py"
        if marker.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find the core repo. Clone it into the working directory "
        "or set REPO_ROOT to the checkout that contains scripts/train.py."
    )


REPO = find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO =", REPO)
print("GPU  =", subprocess.getoutput("nvidia-smi -L") or "none detected")

## Setup — install Python deps

Colab/Kaggle already ship a CUDA torch build in many runtimes. This cell
installs the remaining project dependencies with `pip`. Adjust the torch
index if your host has no GPU wheel preinstalled.

In [ ]:
import subprocess
import sys

# Lightweight extras used by scripts/; torch/torchvision usually preinstalled.
pkgs = [
    "numpy",
    "pillow",
    "pandas",
    "tqdm",
    "pydantic",
    "pyyaml",
    "gdown",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import torch

    print("torch", torch.__version__, "cuda", torch.cuda.is_available())
except ImportError:
    print("torch missing — install a CUDA build matching this host, then re-run")

## Download

Runs `scripts/download-dataset.py` (starter-kit clone + annotation copy +
optional PETA remap). Large image archives still need to be present under
`data/` per the starter kit's own download steps — this script does not
bypass competition licence terms.

In [ ]:
import subprocess
import sys
from pathlib import Path

script = Path("scripts") / "download-dataset.py"
subprocess.check_call([sys.executable, str(script)], cwd=REPO)
print("data/ annotations ready under", REPO / "data")

## Train

Default config is the smoke run (fast). Switch `CONFIG` to
`configs/convnext_baseline.yml` for a full-data train on GPU.

In [ ]:
import subprocess
import sys
from pathlib import Path

CONFIG = Path("configs") / "smoke.yml"  # or configs/convnext_baseline.yml
script = Path("scripts") / "train.py"
subprocess.check_call(
    [sys.executable, str(script), "--config", str(CONFIG)],
    cwd=REPO,
)

## Eval

Writes Track-2 val metrics (mADM / mAP / Rank-k / mINP / mA) next to the run
under `runs/<name>/evaluation.json`.

In [ ]:
import subprocess
import sys
from pathlib import Path

CONFIG = Path("configs") / "smoke.yml"  # keep in sync with Train cell
script = Path("scripts") / "evaluate.py"
subprocess.check_call(
    [sys.executable, str(script), "--config", str(CONFIG)],
    cwd=REPO,
)

## Package

Assembles `build/submission/` from the tracked template + checkpoint, then
zips it for Codabench upload. Download the zip from the host UI when done.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

CONFIG = Path("configs") / "smoke.yml"
# Match runs.name in the config you trained (smoke.yml -> runs/smoke/...)
CHECKPOINT = Path("runs") / "smoke" / "checkpoints" / "best.pt"
OUT = Path("build") / "submission"
ZIP = Path("build") / "submission.zip"

script = Path("scripts") / "package-submission.py"
subprocess.check_call(
    [
        sys.executable,
        str(script),
        "--checkpoint",
        str(CHECKPOINT),
        "--config",
        str(CONFIG),
        "--out",
        str(OUT),
    ],
    cwd=REPO,
)

# Zip archive contents (run.py at archive root), not the folder itself.
if ZIP.exists():
    ZIP.unlink()
shutil.make_archive(str(ZIP.with_suffix("")), "zip", root_dir=OUT)
print("wrote", ZIP.resolve())
print("upload this zip to Codabench; do not commit weights or data")